In [28]:
# !pip install kafka-python

In [ ]:
# from kafka import KafkaConsumer

# consumer = KafkaConsumer(
#     "movielog6",
#     bootstrap_servers="localhost:9092",
#     auto_offset_reset="earliest",  # start from beginning
#     enable_auto_commit=True,
#     group_id="my-group"
# )

# with open("kafka_dump.json", "w", encoding="utf-8") as f:
#     for message in consumer:
#         line = message.value.decode("utf-8")
#         print(line)
#         f.write(line + "\n")


2025-07-27T05:21:08,44425,GET /data/m/mortal+kombat+1995/59.mpg
2025-07-27T05:23:09,44425,GET /data/m/mortal+kombat+1995/61.mpg
2025-07-27T05:25:13,44425,GET /data/m/mortal+kombat+1995/63.mpg
2025-07-27T05:27:19,44425,GET /data/m/mortal+kombat+1995/65.mpg
2025-07-27T05:29:16,44425,GET /data/m/mortal+kombat+1995/67.mpg
2025-07-27T05:31:14,44425,GET /data/m/mortal+kombat+1995/69.mpg
2025-07-27T05:33:09,44425,GET /data/m/mortal+kombat+1995/71.mpg
2025-07-27T05:35:08,44425,GET /data/m/mortal+kombat+1995/73.mpg
2025-07-27T05:37:06,44425,GET /data/m/mortal+kombat+1995/75.mpg
2025-07-27T05:39:08,44425,GET /data/m/mortal+kombat+1995/77.mpg
2025-07-27T05:41:10,44425,GET /data/m/mortal+kombat+1995/79.mpg
2025-07-27T05:43:09,44425,GET /data/m/mortal+kombat+1995/81.mpg
2025-07-27T05:45:10,44425,GET /data/m/mortal+kombat+1995/83.mpg
2025-07-27T05:47:13,44425,GET /data/m/mortal+kombat+1995/85.mpg
2025-07-27T05:49:08,44425,GET /data/m/mortal+kombat+1995/87.mpg
2025-07-27T05:51:08,44425,GET /data/m/mo

KeyboardInterrupt: 

In [30]:
# Install the requests module
# !pip install requests

In [1]:
# import requests

import requests

resp = requests.get("http://128.2.220.241:8080/user/1")  # adjust to actual endpoint
print(resp.json())
resp1 = requests.get("http://128.2.220.241:8080/movie/american+pie+1999")  # adjust to actual endpoint
print(resp1.json())

{'user_id': 1, 'age': 34, 'occupation': 'sales/marketing', 'gender': 'M'}
{'id': 'american+pie+1999', 'tmdb_id': 2105, 'imdb_id': 'tt0163651', 'title': 'American Pie', 'original_title': 'American Pie', 'adult': 'False', 'belongs_to_collection': {'id': 2806, 'name': 'American Pie Collection', 'poster_path': '/twY1eM88fu8CCGVtem5ItlrgeDA.jpg', 'backdrop_path': '/3ptXtaqzAZlMXn3nZxaYke4anmK.jpg'}, 'budget': '11000000', 'genres': [{'id': 35, 'name': 'Comedy'}, {'id': 10749, 'name': 'Romance'}], 'homepage': 'null', 'original_language': 'en', 'overview': 'At a high-school party, four friends find that losing their collective virginity isn\'t as easy as they had thought. But they still believe that they need to do so before college. To motivate themselves, they enter a pact to all "score." by their senior prom.', 'popularity': '18.344227', 'poster_path': '/k40WFAXMRekWEqsjURO3jiWob67.jpg', 'production_companies': [{'name': 'Universal Pictures', 'id': 33}, {'name': 'Summit Entertainment', 'id'

In [32]:
# !pip install tqdm

Dont forget to connect your localhost to the remote kafka server
```bash
 ssh -L 9092:localhost:9092 tunnel@128.2.220.241 -NT --
```

In [2]:
#CONSUMING KAFKA LOGS AND STORING THEM IN FILES

from kafka import KafkaConsumer, TopicPartition
from tqdm import tqdm
import json

topic = "movielog6"
bootstrap = "localhost:9092"

consumer = KafkaConsumer(
    bootstrap_servers=bootstrap,
    auto_offset_reset="earliest",
    enable_auto_commit=False,
    group_id=None,   # ensures we don't commit offsets
)

# assume single-partition topic (common for course setup)
tp = TopicPartition(topic, 0)
consumer.assign([tp])

# find total messages in backlog
begin = consumer.beginning_offsets([tp])[tp]
end = consumer.end_offsets([tp])[tp]
total = end - begin
print(f"Dumping ~{total} messages from Kafka...")

# open output files
watch_f = open("watch_events.jsonl", "w", encoding="utf-8")
rating_f = open("rating_events.jsonl", "w", encoding="utf-8")
rec_f = open("recommendation_events.jsonl", "w", encoding="utf-8")

with tqdm(total=total) as pbar:
    for msg in consumer:
        line = msg.value.decode("utf-8").strip()

        # classify event type
        if "recommendation request" in line:
            rec_f.write(json.dumps({"raw": line}) + "\n")
        elif "GET /data/m/" in line:
            watch_f.write(json.dumps({"raw": line}) + "\n")
        elif "GET /rate/" in line:
            rating_f.write(json.dumps({"raw": line}) + "\n")

        pbar.update(1)

        # stop after backlog
        if msg.offset + 1 >= end:
            break

# cleanup
watch_f.close()
rating_f.close()
rec_f.close()
consumer.close()
print("✅ Dump complete")


Dumping ~136335791 messages from Kafka...


100%|██████████| 136335791/136335791 [2:02:11<00:00, 18597.05it/s] 

✅ Dump complete


In [3]:
# Now, let's read back a few records from one of the files to verify

import json

file_path = "watch_events.jsonl"

with open(file_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 10:  # only look at first 10 records
            break
        record = json.loads(line)
        print(record)


{'raw': '2025-07-25T15:14:41,119750,GET /data/m/capote+2005/0.mpg'}
{'raw': '2025-07-25T15:15:40,119750,GET /data/m/capote+2005/1.mpg'}
{'raw': '2025-07-25T15:17:45,119750,GET /data/m/capote+2005/3.mpg'}
{'raw': '2025-07-25T15:19:44,119750,GET /data/m/capote+2005/5.mpg'}
{'raw': '2025-07-25T15:21:42,119750,GET /data/m/capote+2005/7.mpg'}
{'raw': '2025-07-25T15:23:34,62937,GET /data/m/american+pie+1999/0.mpg'}
{'raw': '2025-07-25T15:24:36,62937,GET /data/m/american+pie+1999/1.mpg'}
{'raw': '2025-07-25T15:25:36,62937,GET /data/m/american+pie+1999/2.mpg'}
{'raw': '2025-07-25T15:26:39,62937,GET /data/m/american+pie+1999/3.mpg'}
{'raw': '2025-07-25T15:27:45,62937,GET /data/m/american+pie+1999/4.mpg'}


In [4]:
# Extract unique user IDs and movie IDs from watch events

import json
from tqdm import tqdm
import os

watch_file = "watch_events.jsonl"  # your file path

user_ids = set()
movie_ids = set()

# get total number of lines (for tqdm progress bar)
def count_lines(filename):
    with open(filename, "r", encoding="utf-8", errors="ignore") as f:
        return sum(1 for _ in f)

total_lines = count_lines(watch_file)

with open(watch_file, "r", encoding="utf-8", errors="ignore") as f:
    for line in tqdm(f, total=total_lines, desc="Processing watch events"):
        try:
            record = json.loads(line)
            raw = record["raw"]

            # Example: "2025-07-25T15:14:41,119750,GET /data/m/capote+2005/0.mpg"
            parts = raw.split(",")
            if len(parts) < 3:
                continue

            user_id = parts[1].strip()
            path = parts[2].strip()

            # Extract movie_id between /m/ and the next /
            movie_id = path.split("/data/m/")[1].split("/")[0]

            user_ids.add(int(user_id))
            movie_ids.add(movie_id)

        except Exception:
            continue

# Convert sets to sorted lists
user_ids = sorted(user_ids)
movie_ids = sorted(movie_ids)

# print("Sample user_ids:", user_ids[:10])
# print("Sample movie_ids:", movie_ids[:10])
print(f"Total unique users: {len(user_ids)}")
print(f"Total unique movies: {len(movie_ids)}")


Processing watch events: 100%|██████████| 135394674/135394674 [08:35<00:00, 262678.92it/s]


Total unique users: 144736
Total unique movies: 26867


In [5]:
with open("unique_users.csv", "w", encoding="utf-8") as f:
    for uid in user_ids:
        f.write(f"{uid}\n")

with open("unique_movies.csv", "w", encoding="utf-8") as f:
    for mid in movie_ids:
        f.write(f"{mid}\n")

In [ ]:
# !pip install pandas

  Using cached pandas-2.3.2-cp310-cp310-win_amd64.whl (11.3 MB)
  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl (12.9 MB)
  Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)



[notice] A new release of pip available: 22.2.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
# Now, fetch user and movie data from the API and save to CSVs

import requests
import pandas as pd
from tqdm import tqdm

# ------------------------
# Configuration
# ------------------------
API_BASE = "http://128.2.220.241:8080"
# USER_IDS = [1, 2, 3, 4, 5]          # replace with the list of user IDs you have
# MOVIE_IDS = ["Inception_2010", "Interstellar_2014"]  # replace with your movie IDs

# ------------------------
# Fetch User Data
# ------------------------
user_records = []

for uid in tqdm(user_ids, desc="Fetching user data"):
    try:
        resp = requests.get(f"{API_BASE}/user/{uid}", timeout=5)
        if resp.status_code == 200:
            user_records.append(resp.json())
        else:
            print(f"Warning: Failed to fetch user {uid}, status {resp.status_code}")
    except Exception as e:
        print(f"Error fetching user {uid}: {e}")

# Save to CSV
users_df = pd.DataFrame(user_records)
users_df.to_csv("users.csv", index=False)
print(f"Saved {len(users_df)} users to users.csv")

# ------------------------
# Fetch Movie Data
# ------------------------
movie_records = []

for mid in tqdm(movie_ids, desc="Fetching movie data"):
    try:
        resp = requests.get(f"{API_BASE}/movie/{mid}", timeout=5)
        if resp.status_code == 200:
            movie_records.append(resp.json())
        else:
            print(f"Warning: Failed to fetch movie {mid}, status {resp.status_code}")
    except Exception as e:
        print(f"Error fetching movie {mid}: {e}")

# Save to CSV
movies_df = pd.DataFrame(movie_records)
movies_df.to_csv("movies.csv", index=False)
print(f"Saved {len(movies_df)} movies to movies.csv")


Fetching user data: 100%|██████████| 144736/144736 [1:55:14<00:00, 20.93it/s]  


Saved 144736 users to users.csv


Fetching movie data: 100%|██████████| 26867/26867 [35:17<00:00, 12.69it/s]


Saved 26867 movies to movies.csv


In [14]:
import json
import pandas as pd
from tqdm import tqdm
from collections import defaultdict

# -----------------------
# File paths (update these)
# -----------------------
watch_file = "watch_events.jsonl"
rating_file = "rating_events.jsonl"
# movies_file = "movies.csv"

# -----------------------
# Build Aggregated Interactions Table
# -----------------------
def build_interactions_table(watch_file, rating_file,
                             output_csv="interactions_table.csv"):

    # --- Load movies (for reference if needed)
    # df_movies = pd.read_csv(movies_file)

    # --- Track watch info
    watch_data = defaultdict(lambda: {
        "total_minutes": 0,
        "last_watch_time": None
    })

    # Count lines for tqdm
    def count_lines(filename):
        with open(filename, "r", encoding="utf-8", errors="ignore") as f:
            return sum(1 for _ in f)

    total_watch_lines = count_lines(watch_file)
    total_rating_lines = count_lines(rating_file)

    # --- Process watch events
    with open(watch_file, "r", encoding="utf-8", errors="ignore") as f:
        for line in tqdm(f, total=total_watch_lines, desc="Processing watch events"):
            try:
                record = json.loads(line.strip())
                raw = record["raw"]
                parts = raw.split(",")
                if len(parts) < 3:
                    continue

                timestamp, user_id, path = parts[0], parts[1], parts[2]
                movie_id = path.split("/data/m/")[1].split("/")[0]

                key = (int(user_id), movie_id)

                # accumulate minutes watched
                watch_data[key]["total_minutes"] += 1
                watch_data[key]["last_watch_time"] = timestamp
            except:
                continue

    # --- Track ratings
    ratings = {}
    with open(rating_file, "r", encoding="utf-8", errors="ignore") as f:
        for line in tqdm(f, total=total_rating_lines, desc="Processing rating events"):
            try:
                record = json.loads(line.strip())
                raw = record["raw"]
                parts = raw.split(",")
                if len(parts) < 3:
                    continue

                timestamp, user_id, path = parts[0], parts[1], parts[2]

                # Example: GET /rate/the+lord+of+the+rings+2003=3
                if "/rate/" not in path:
                    continue

                rate_part = path.split("/rate/")[1]
                if "=" not in rate_part:
                    continue

                movie_id, rating = rate_part.split("=")
                # movie_id = movie_id.replace("+", " ")
                rating = int(rating)

                key = (int(user_id), movie_id)
                ratings[key] = (rating, timestamp)  # keep last rating
            except:
                continue

    # --- Build final interactions table
    rows = []
    for key, watch_info in watch_data.items():
        user_id, movie_id = key
        total_minutes = watch_info["total_minutes"]
        last_watch_time = watch_info["last_watch_time"]

        rating, _ = ratings.get(key, (None, None))

        rows.append({
            "user_id": user_id,
            "movie_id": movie_id,
            "total_minutes": total_minutes,
            "rating": rating,
            "last_watch_time": last_watch_time
        })

    df_interactions = pd.DataFrame(rows)
    df_interactions.to_csv(output_csv, index=False)
    print(f"✅ Interactions table saved as {output_csv}")
    return df_interactions

# -----------------------
# Run
# -----------------------
build_interactions_table(watch_file, rating_file)


Processing rating events: 100%|██████████| 876127/876127 [00:02<00:00, 308199.96it/s]


✅ Interactions table saved as interactions_table.csv


,user_id,movie_id,total_minutes,rating,last_watch_time
0,119750,capote+2005,61,NaN,2025-07-25T17:15:26
1,62937,american+pie+1999,47,NaN,2025-07-25T16:37:14
2,2432,ace+ventura+when+nature+calls+1995,18,NaN,2025-07-25T18:02:20
3,104487,shrek+2001,74,NaN,2025-07-25T19:34:08
4,39157,star+wars+episode+ii+-+attack+of+the+clones+2002,63,NaN,2025-07-25T22:54:35
...,...,...,...,...,...
2504325,11902,dear+pillow+2004,1,NaN,2025-09-23T19:08:14
2504326,27298,lucky+night+1939,1,NaN,2025-09-23T19:08:15
2504327,52834,up+2009,1,NaN,2025-09-23T19:08:16
2504328,44271,the+shining+1980,1,NaN,2025-09-23T19:08:16


# Why do the rows drastically reduce from the kafka logs to the table created?

Watch events (135M lines)

Each line is a minute watched.

We aggregate all those minutes into a single row per (user_id, movie_id).

Example:

user 96702 watched "independence+day+1996"
→ 48 minutes watched
→ 1 row

So 135M raw watch events collapse into far fewer unique (user_id, movie_id) pairs.

Final table (~2.5M rows)

Each row corresponds to one unique (user, movie) interaction.

That’s why the reduction is so dramatic — we went from events → interactions.

In [5]:
import pandas as pd
import json
import ast
import numpy as np

movies_file = "movies.csv"
output_file = "movies_table_clean.csv"

# Load the movies CSV
df = pd.read_csv(movies_file)

# Columns with embedded JSON
json_cols = [
    "belongs_to_collection",
    "genres",
    "production_companies",
    "production_countries",
    "spoken_languages"
]

def safe_parse(val):
    """Safely parse JSON-like strings, return None if empty or invalid."""
    if pd.isna(val) or str(val).strip() in ["", "null", "None"]:
        return None
    try:
        return json.loads(val)
    except:
        try:
            return ast.literal_eval(val)  # fallback if single quotes
        except:
            return None

# Process each JSON column
for col in json_cols:
    df[col] = df[col].apply(safe_parse)

# Extract useful info
df["collection_name"] = df["belongs_to_collection"].apply(
    lambda x: x.get("name") if isinstance(x, dict) else None
)
df["genres_list"] = df["genres"].apply(
    lambda x: [g.get("name") for g in x] if isinstance(x, list) else None
)
df["production_companies_list"] = df["production_companies"].apply(
    lambda x: [c.get("name") for c in x] if isinstance(x, list) else None
)
df["production_countries_list"] = df["production_countries"].apply(
    lambda x: [c.get("name") for c in x] if isinstance(x, list) else None
)
df["spoken_languages_list"] = df["spoken_languages"].apply(
    lambda x: [l.get("name") for l in x] if isinstance(x, list) else None
)

# Drop original messy JSON columns
df_clean = df.drop(columns=json_cols)

# Save cleaned CSV
df_clean.to_csv(output_file, index=False)
print(f"✅ Cleaned movies table saved as {output_file}")


✅ Cleaned movies table saved as movies_table_clean.csv
